# Chapter 1: Common Conventions and API Elements of scikit-learn

*Summary notebook for **scikit-learn Cookbook, Third Edition** (John Sukup, Packt 2025) — Chapter 1.*

This chapter has no heavy modelling. Its purpose is to teach the **vocabulary and conventions** that every later chapter relies on: *estimators*, *transformers*, *pipelines*, and the methods `fit()`, `predict()`, `transform()`.

**Chapter at a glance**

| Recipe | Core idea |
|:--|:--|
| Design philosophy | Consistency, simplicity, modularity, reusability |
| Estimators | Objects that learn from data via `fit()` |
| Transformers | Estimators that change data via `transform()` |
| Custom estimators | Subclass `BaseEstimator` + mixins |
| Pipelines | Chain steps into one object; prevents leakage |
| Attributes & methods | `coef_`, `intercept_`, `score()` |
| Hyperparameter tuning | `get_params()`, `set_params()`, grid/random search |
| Metadata | Estimator tags and metadata routing |
| Best practices | Uniform API, preprocess, pipeline, CV, tune |

In [1]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import sklearn
print(f"scikit-learn version: {sklearn.__version__}")
print(f"NumPy version: {np.__version__}")

scikit-learn version: 1.9.1
NumPy version: 2.5.3


> **Note on versions.** The book targets scikit-learn **1.5+**. The code below runs on any recent version; the version printed above is the one actually installed in this session. A few minor API details (e.g. `multi_class` in `LogisticRegression`, used in Chapter 6) changed in newer releases — such cases are flagged where they appear.

## 1. scikit-learn's Design Philosophy

### Summary
The library is built on four principles:

1. **Consistency** – every algorithm exposes the same small set of methods (`fit`, `predict`, `transform`).
2. **Simplicity** – sensible default hyperparameters let you get a result immediately.
3. **Modularity** – estimators, transformers and pipelines are independent blocks that can be chained.
4. **Reusability** – a component written once (even your own) plugs into any other workflow.

Consequence: swapping `LogisticRegression()` for `RandomForestClassifier()` is a **one-line change**.

> *Naming trivia:* the correct spelling is lowercase **scikit-learn** (pronounced "sy-kit"), where *sci* = *science*.

### Theory explained
**Why does a uniform API matter?** In software-design terms, scikit-learn applies the *interface segregation / duck typing* idea: any object that has `fit` (and `predict` or `transform`) can be used in any scikit-learn tool (`Pipeline`, `GridSearchCV`, `cross_val_score`). The tools never need to know *which* algorithm is inside — only that it "quacks" like an estimator. This is what lets cross-validation, tuning, and pipelines work with 100+ different algorithms with no special-casing.

## 2. Understanding Estimators

### Summary
An **estimator** is any object that learns from data. Its essential method is `fit()`. Supervised estimators add `predict()`.

The universal three-step pattern:
```python
model = Estimator(hyperparameters)   # 1. create
model.fit(X_train, y_train)          # 2. learn
model.predict(X_new)                 # 3. use
```

### Theory explained — Linear Regression
Given a feature matrix $\mathbf{X}\in\mathbb{R}^{n\times p}$ (n samples, p features) and target $\mathbf{y}\in\mathbb{R}^n$, linear regression assumes

$$\hat y = \mathbf{w}^T\mathbf{x} + b$$

and chooses $\mathbf{w}, b$ to minimise the **sum of squared residuals** (Ordinary Least Squares, OLS):

$$\min_{\mathbf{w},b}\sum_{i=1}^{n}\bigl(y_i-(\mathbf{w}^T\mathbf{x}_i+b)\bigr)^2$$

This has a closed-form solution, the **normal equations** (with the intercept absorbed into $\mathbf{X}$ as a column of ones):

$$\mathbf{w}=(\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y}$$

*Intuition:* we pick the line whose vertical distances to all points are, in squared total, as small as possible.

In [2]:
from sklearn.linear_model import LinearRegression

X = np.array([[1], [2], [3], [4], [5]])   # feature matrix (5 samples, 1 feature)
y = np.array([1, 2, 3, 3.5, 5])           # target values

model = LinearRegression()
model.fit(X, y)

X_new = np.array([[6], [7]])
predictions = model.predict(X_new)
print("Predictions for X = [6, 7]:", predictions)

Predictions for X = [6, 7]: [5.75 6.7 ]


**Reading the output.** The predictions are $5.75$ for $x=6$ and $6.70$ for $x=7$. They come from the learned line $\hat y = 0.95x+0.05$ (the coefficients are inspected in section 6): $0.95\times6+0.05=5.75$.

### `fit_predict()` — shortcut for unsupervised learning

`fit_predict()` = `fit()` + `predict()` on the **same data**. It is mainly used for **clustering**, where there is no target $\mathbf{y}$ and you want the labels of the training data itself.

**K-Means theory.** K-Means partitions $n$ points into $K$ clusters by minimising the **within-cluster sum of squares** (WCSS, also called *inertia*):

$$\min_{\{C_k\}}\sum_{k=1}^{K}\sum_{\mathbf{x}_i\in C_k}\lVert\mathbf{x}_i-\boldsymbol\mu_k\rVert^2,\qquad \boldsymbol\mu_k=\frac1{|C_k|}\sum_{\mathbf{x}_i\in C_k}\mathbf{x}_i$$

Algorithm (Lloyd's): (1) initialise K centroids → (2) assign each point to the nearest centroid → (3) move each centroid to the mean of its points → repeat 2–3 until nothing changes.

In [3]:
from sklearn.cluster import KMeans

X = np.array([[1], [2], [3], [4], [5]])

kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
labels = kmeans.fit_predict(X)
print("Cluster labels:", labels)
print("Cluster centers:", kmeans.cluster_centers_.flatten())

Cluster labels: [0 0 0 1 1]
Cluster centers: [2.  4.5]


**Reading the output.** The points split into two groups and each center is the mean of its group. (Which group gets label 0 vs 1 is arbitrary — only the *grouping* is meaningful.)

**When to use which**
- *Supervised:* `fit(X_train, y_train)` then `predict(X_test)` separately — train and test must stay distinct.
- *Unsupervised:* `fit_predict(X)` is convenient when you want labels for the training data itself. Train/test splits are still possible if you want to test generalisation.

## 3. Transformers and `transform()`

### Summary
A **transformer** is an estimator that *modifies data* instead of predicting. Methods:

- `fit(X)` – learn parameters from data (e.g. mean and std)
- `transform(X)` – apply what was learned
- `fit_transform(X)` – both in one step

### Theory explained — Standardization (z-score)
$$z_{ij}=\frac{x_{ij}-\mu_j}{\sigma_j}$$

Each feature $j$ is shifted to mean 0 and scaled to standard deviation 1. **Why?** Distance-based and gradient-based algorithms (KNN, SVM, logistic regression, neural nets, PCA) are sensitive to feature scale: a feature in the thousands (income) would dominate one in decimals (GPA). Standardization puts features on equal footing.

> Note: `StandardScaler` uses the **population** standard deviation (divide by $n$, not $n-1$).

In [4]:
from sklearn.preprocessing import StandardScaler

X = np.array([[1, 2], [3, 4], [5, 6]])

scaler = StandardScaler()
scaler.fit(X)
X_scaled = scaler.transform(X)

print("Original data:\n", X)
print("\nScaled data:\n", X_scaled)
print("\nMean of each feature (learned):", scaler.mean_)
print("Std of each feature (learned):", scaler.scale_)

Original data:
 [[1 2]
 [3 4]
 [5 6]]

Scaled data:
 [[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]

Mean of each feature (learned): [3. 4.]
Std of each feature (learned): [1.63299316 1.63299316]


**Check by hand:** $z_{11}=\dfrac{1-3}{1.633}=-1.2247$ ✔.

### The critical rule: avoid data leakage
- `fit_transform()` on **training** data only.
- `transform()` (never `fit`) on **test** data.

Re-fitting on test data computes a *different* mean/std, so train and test would be on different scales — and, worse, test information leaks into preprocessing.

In [5]:
X_scaled = StandardScaler().fit_transform(X)
print(X_scaled)
print("Column means:", X_scaled.mean(axis=0))
print("Column stds :", X_scaled.std(axis=0))

[[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]
Column means: [0. 0.]
Column stds : [1. 1.]


## 4. Custom Estimators and Transformers

### Summary
You can write your own components by subclassing `BaseEstimator` plus **mixins**:

| Mixin | Adds |
|:--|:--|
| `ClassifierMixin` | `score()` = accuracy |
| `RegressorMixin` | `score()` = $R^2$ |
| `TransformerMixin` | `fit_transform()` |

### Theory explained — mixins
A **mixin** is a small class that supplies *one* capability and is combined with others through multiple inheritance, instead of building a deep class tree. `TransformerMixin` simply implements `fit_transform(X) = self.fit(X).transform(X)` — which is why you only need to write `fit` and `transform`.

**Conventions every custom estimator should follow**
1. `__init__` only *stores* hyperparameters (no computation) → enables `get_params()/set_params()` and `clone()`.
2. `fit()` returns `self` → enables chaining.
3. Learned attributes end with an underscore (`n_features_in_`).
4. `check_is_fitted()` guards `transform/predict`.

In [6]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted

class ValueClipper(BaseEstimator, TransformerMixin):
    "Clips all feature values to the [lower, upper] range."

    def __init__(self, lower=0.0, upper=1.0):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        self.n_features_in_ = np.asarray(X).shape[1]
        self.is_fitted_ = True
        return self

    def transform(self, X):
        check_is_fitted(self, 'is_fitted_')
        return np.clip(np.asarray(X), self.lower, self.upper)

X = np.array([[0.5, -0.3], [1.5, 0.7], [0.2, 2.1]])
clipper = ValueClipper(lower=0.0, upper=1.0)
print(clipper.fit_transform(X))      # fit_transform comes free from TransformerMixin
print("Parameters:", clipper.get_params())

[[0.5 0. ]
 [1.  0.7]
 [0.2 1. ]]
Parameters: {'lower': 0.0, 'upper': 1.0}


Values outside $[0,1]$ were moved to the nearest boundary ($-0.3\to0$, $1.5\to1$, $2.1\to1$). Because `ValueClipper` obeys the conventions, it can go straight into a `Pipeline` or `GridSearchCV` (tuning `lower`/`upper`).

## 5. Pipelines and Workflow Automation

### Summary
A **Pipeline** chains transformers and (optionally) a final estimator into one object. `fit()` runs `fit_transform()` on each transformer then `fit()` on the last step; `predict()`/`score()` run only `transform()` on the transformers.

### Theory explained
For transformers $T_1,\dots,T_K$ and final estimator $E$:

$$\hat y=E\bigl(T_K(\dots T_2(T_1(\mathbf X)))\bigr)$$

Benefits: **no leakage** (preprocessing is learned only from the data passed to `fit`), **reproducibility**, **one deployable artifact**, and the whole thing can be tuned by `GridSearchCV`.

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=200, n_features=5, n_informative=3,
                           n_redundant=1, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])
pipe.fit(X_train, y_train)

print(f"Training accuracy: {pipe.score(X_train, y_train):.4f}")
print(f"Test accuracy:     {pipe.score(X_test, y_test):.4f}")
print("Pipeline steps:   ", [name for name, _ in pipe.steps])

Training accuracy: 0.9267
Test accuracy:     0.8800
Pipeline steps:    ['scaler', 'classifier']


**Reading the output.** Train and test accuracy are close, so the model is not badly overfit (gap = difference of accuracies, expressed as a proportion, not "percentage points").

### MLOps note
Production ML is a cycle, not a line: **Continuous Training (CT)**, **CI/CD**, and **monitoring** (drift detection). *"There is no such thing as a model that works well forever."* A `Pipeline` is the natural unit to version and deploy (e.g. with `joblib`, MLflow, Kubeflow) — Chapter 13 covers this.

## 6. Common Attributes and Methods

### Summary
Learned values are stored in attributes with a **trailing underscore** (available only after `fit`): `coef_`, `intercept_`, `feature_importances_`, `cluster_centers_`, …

`score()` returns a default metric: **accuracy** for classifiers, **$R^2$** for regressors.

### Theory explained — $R^2$
$$R^2=1-\frac{\sum_i(y_i-\hat y_i)^2}{\sum_i(y_i-\bar y)^2}=1-\frac{SS_{res}}{SS_{tot}}$$
It is the fraction of the target's variance explained by the model (1 = perfect, 0 = no better than predicting the mean). Caveat: it **never decreases when you add features**, even noise. The **adjusted** version penalises this:

$$R^2_{adj}=1-\frac{(1-R^2)(n-1)}{n-p-1}$$

In [8]:
X = np.array([[1], [2], [3], [4], [5]])
y = np.array([1, 2, 3, 3.5, 5])

model = LinearRegression().fit(X, y)
print("Coefficients (w):", model.coef_)
print("Intercept (b):   ", model.intercept_)
print(f"R-squared:         {model.score(X, y):.6f}")

y_hat = model.coef_[0] * 3 + model.intercept_
print(f"\nManual prediction for X=3: {y_hat:.2f}   (actual y = {y[2]})")

Coefficients (w): [0.95]
Intercept (b):    0.04999999999999938
R-squared:         0.980978

Manual prediction for X=3: 2.90   (actual y = 3.0)


The fitted line is $\hat y=0.95x+0.05$ with $R^2\approx0.981$: about 98% of the variance is explained. It is not 1.0 because the point $(4,\,3.5)$ lies below the straight trend.

## 7. Hyperparameter Tuning with Search Methods

### Summary
**Parameters** are learned during `fit` (e.g. `coef_`). **Hyperparameters** are chosen *before* training and tell the model *how to learn* (e.g. `n_estimators`, `C`, `max_depth`).

Two ways to set them:
1. Manually: `set_params()` / `get_params()`.
2. Automatically: `GridSearchCV` (exhaustive), `RandomizedSearchCV` (random sample), plus successive-halving variants (`HalvingGridSearchCV`, `HalvingRandomSearchCV`).

In [9]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()
model.set_params(n_estimators=100, max_depth=10, random_state=42)

for key, value in sorted(model.get_params().items()):
    print(f"  {key}: {value}")

  bootstrap: True
  ccp_alpha: 0.0
  class_weight: None
  criterion: gini
  max_depth: 10
  max_features: sqrt
  max_leaf_nodes: None
  max_samples: None
  min_impurity_decrease: 0.0
  min_samples_leaf: 1
  min_samples_split: 2
  min_weight_fraction_leaf: 0.0
  monotonic_cst: None
  n_estimators: 100
  n_jobs: None
  oob_score: False
  random_state: 42
  verbose: 0
  warm_start: False


### Theory explained — key Random Forest hyperparameters
- `n_estimators` – number of trees. More trees → lower variance, linearly higher cost.
- `max_depth` – maximum tree depth; limits complexity (regularisation).
- `max_features='sqrt'` – each split considers only $\sqrt p$ random features, which **decorrelates** the trees (the source of the ensemble's variance reduction).
- `criterion='gini'` – split quality. **Gini impurity**:

$$G=1-\sum_{c=1}^{C}p_c^2$$

where $p_c$ is the share of class $c$ in the node. Pure node → $G=0$; a 50/50 binary node → $G=0.5$.
- `random_state` – reproducibility.

## 8. Metadata: Tags and Metadata Routing

### Summary
- **Estimator tags** describe capabilities (multi-output? handles NaN? requires positive input?). scikit-learn infrastructure reads them instead of trial-and-error.
- **Metadata routing** controls how extra per-sample information (`sample_weight`, `groups`, …) flows through meta-estimators.
  - **Routers** (`Pipeline`, `GridSearchCV`) forward metadata.
  - **Consumers** (estimators, scorers) use it.

*Example:* in an imbalanced problem, route `sample_weight` to the classifier step but not to the scaler.

In [10]:
lr = LogisticRegression()
tags = lr.__sklearn_tags__()
print("Estimator:", type(lr).__name__)
print("estimator_type:", tags.estimator_type)
print("non_deterministic:", tags.non_deterministic)
print("array_api_support:", tags.array_api_support)

Estimator: LogisticRegression
estimator_type: classifier
non_deterministic: False
array_api_support: True


Tags act as a **contract** between an estimator and the scikit-learn machinery. You will mainly touch them when writing custom estimators.

## 9. Best Practices for API Usage

1. **Uniform API** – learn `fit → predict/transform` once.
2. **Preprocess** (scale, encode, impute) before modelling.
3. **Use pipelines** for multi-step workflows (no leakage, reproducible).
4. **Cross-validate** – never judge a model on training data only.
5. **Tune hyperparameters** systematically (`GridSearchCV` / `RandomizedSearchCV`).
6. **Inspect internals** (`coef_`, `feature_importances_`) for interpretability.

---
## Chapter 1 Summary

| Concept | Key methods | Role |
|:--|:--|:--|
| Estimator | `fit()`, `predict()` | Learn and predict |
| Transformer | `fit()`, `transform()`, `fit_transform()` | Modify data |
| Pipeline | `fit()`, `predict()`, `score()` | Chain steps |
| Attributes | `coef_`, `intercept_` | Inspect what was learned |
| Evaluation | `score()` | Quick metric |
| Hyperparameters | `get_params()`, `set_params()` | Configure |
| Metadata | `__sklearn_tags__()` | Describe capabilities |